In [ ]:
# ====================== 导入库 & 全局配置 ======================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats.mstats import winsorize
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# 绘图中文设置
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 数据根目录 = 当前脚本所在目录
DATA_ROOT = Path(__file__).parent

# ========== 可调开关 ==========
# 做月度趋势分析建议开启，做总量统计建议关闭
FILTER_ABNORMAL_MONTH = True
abnormal_months = ["2016-09", "2016-12", "2018-09", "2018-10"]

# 做均值/分布分析建议开启，算总额建议关闭
# 注意：limits = (下端截断比例, 上端截断比例)，例如 (0.005, 0.005) = 上下各去掉0.5%
ENABLE_WINSORIZE = True
WINSORIZE_LIMITS = (0.005, 0.005)
winsorize_cols = ["order_value", "delivery_hours", "price_total"]

# ====================== 加载清洗好的中间表 ======================
print("="*50)
print("加载清洗后的数据表...")
print("="*50)

df_main = pd.read_csv(
    DATA_ROOT / "df_main_wide.csv",
    parse_dates=["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
)
df_item = pd.read_csv(DATA_ROOT / "df_item_wide.csv", parse_dates=["order_purchase_timestamp"])
rfm = pd.read_csv(DATA_ROOT / "rfm_base.csv", parse_dates=["last_purchase_time"])
geo = pd.read_csv(DATA_ROOT / "geolocation_clean.csv")

print(f"✅ 订单宽表：{len(df_main):,} 行 × {len(df_main.columns)} 列")
print(f"✅ 商品宽表：{len(df_item):,} 行 × {len(df_item.columns)} 列")
print(f"✅ RFM基础表：{len(rfm):,} 位用户")
print(f"✅ 地理维度表：{len(geo):,} 个邮编区域\n")

# ====================== 生成分析专用数据集 ======================
df_analysis = df_main.copy()

if FILTER_ABNORMAL_MONTH:
    df_analysis = df_analysis[
        ~df_analysis["order_purchase_timestamp"].dt.strftime("%Y-%m").isin(abnormal_months)
    ].reset_index(drop=True)
    print(f"已过滤异常月份，剩余订单：{len(df_analysis):,}")

# 先打印原始配送时长统计，验证数据真实性
print("\n===== 原始配送时长校验 =====")
delivery_raw = df_analysis["delivery_hours"].dropna() / 24
print(f"均值：{delivery_raw.mean():.2f} 天")
print(f"中位数：{delivery_raw.median():.2f} 天")
print(f"95分位数：{delivery_raw.quantile(0.95):.2f} 天")
print(f"最大值：{delivery_raw.max():.2f} 天")

if ENABLE_WINSORIZE:
    for col in winsorize_cols:
        if col in df_analysis.columns:
            df_analysis[f"{col}_wins"] = winsorize(df_analysis[col], limits=WINSORIZE_LIMITS)
    print(f"\n已开启异常值缩尾，上下截断比例：{WINSORIZE_LIMITS}")

# ====================== 模块1：整体运营概览 ======================
print("\n" + "="*50)
print("一、整体运营概览")
print("="*50)

total_orders = len(df_analysis)
total_gmv = df_analysis["order_value"].sum()
avg_order_value = df_analysis["order_value"].mean()
delay_rate = df_analysis["is_delay"].mean()

# 差评率口径明确：基于有评价订单
total_with_review = df_analysis["is_bad_review"].notna().sum()
bad_review_rate = df_analysis["is_bad_review"].mean()

print(f"总订单量：{total_orders:,} 单")
print(f"总GMV：{total_gmv:,.2f} 雷亚尔")
print(f"平均客单价：{avg_order_value:.2f} 雷亚尔")
print(f"整体配送延迟率：{delay_rate*100:.2f}%")
print(f"有评价订单数：{total_with_review:,}")
print(f"整体差评率（≤3分，基于有评价订单）：{bad_review_rate*100:.2f}%")

# 月度趋势表 —— 高版本Pandas兼容写法 ME = Month End
df_monthly = df_analysis.set_index("order_purchase_timestamp").resample("ME").agg(
    订单量=("order_id", "count"),
    总GMV=("order_value", "sum"),
    客单价=("order_value", "mean"),
    延迟率=("is_delay", "mean")
).reset_index()

# 月度订单量趋势图
plt.figure(figsize=(12, 5))
plt.plot(df_monthly["order_purchase_timestamp"], df_monthly["订单量"], marker='o', linewidth=2, color='#2c3e50')
plt.title("月度订单量变化趋势", fontsize=14)
plt.xlabel("月份", fontsize=12)
plt.ylabel("订单数量", fontsize=12)
plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(DATA_ROOT / "月度订单趋势.png", dpi=300, bbox_inches='tight')
plt.show()

# 月度GMV + 客单价双轴趋势图
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(df_monthly["order_purchase_timestamp"], df_monthly["总GMV"],
         color='#e74c3c', marker='o', linewidth=2, label="月度GMV")
ax2.plot(df_monthly["order_purchase_timestamp"], df_monthly["客单价"],
         color='#3498db', marker='s', linewidth=2, linestyle='--', label="客单价")

ax1.set_xlabel("月份", fontsize=12)
ax1.set_ylabel("GMV（雷亚尔）", fontsize=12, color='#e74c3c')
ax2.set_ylabel("客单价（雷亚尔）", fontsize=12, color='#3498db')
ax1.tick_params(axis='y', labelcolor='#e74c3c')
ax2.tick_params(axis='y', labelcolor='#3498db')
plt.title("月度 GMV 与客单价趋势", fontsize=14)
fig.legend(loc="upper left", bbox_to_anchor=(0.1, 0.95))
plt.tight_layout()
plt.savefig(DATA_ROOT / "月度GMV与客单价趋势.png", dpi=300, bbox_inches='tight')
plt.show()

# ====================== 模块2：商品品类TOP10分析 ======================
print("\n" + "="*50)
print("二、商品品类销量TOP10")
print("="*50)

category_sales = df_item.groupby("product_category_name_english").agg(
    销量=("order_item_id", "count"),
    销售额=("item_total_value", "sum"),
    平均单价=("price", "mean")
).reset_index().sort_values("销量", ascending=False).head(10)
print(category_sales.round(2).to_string(index=False))

plt.figure(figsize=(12, 6))
sns.barplot(data=category_sales, x="销量", y="product_category_name_english", palette="viridis")
plt.title("商品品类销量TOP10", fontsize=14)
plt.ylabel("商品品类（英文）", fontsize=12)
plt.xlabel("销量", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_ROOT / "品类销量TOP10.png", dpi=300, bbox_inches='tight')
plt.show()

# ====================== 模块3：物流时效分析 ======================
print("\n" + "="*50)
print("三、物流时效分析")
print("="*50)

delivery_days = df_analysis["delivery_hours_wins"] / 24
print(f"缩尾后平均配送时长：{delivery_days.mean():.2f} 天")
print(f"缩尾后配送时长中位数：{delivery_days.median():.2f} 天")
print(f"缩尾后最长配送时长：{delivery_days.max():.2f} 天")

# 极端延迟订单分析（不缩尾）
extreme_threshold = df_analysis["delivery_hours"].quantile(0.995) / 24
extreme_orders = df_analysis[df_analysis["delivery_hours"]/24 > extreme_threshold]
print(f"\n极端延迟订单（>{extreme_threshold:.0f}天）：{len(extreme_orders)} 单")
print("极端订单的州分布 TOP5：")
print(extreme_orders["customer_state"].value_counts().head())

plt.figure(figsize=(12, 5))
sns.histplot(delivery_days, bins=60, kde=True, color='#27ae60', alpha=0.7)
plt.axvline(delivery_days.mean(), color='red', linestyle='--', linewidth=2, label=f'均值：{delivery_days.mean():.1f}天')
plt.title("订单配送时长分布", fontsize=14)
plt.xlabel("配送天数", fontsize=12)
plt.ylabel("订单数量", fontsize=12)
plt.legend()
plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(DATA_ROOT / "配送时长分布.png", dpi=300, bbox_inches='tight')
plt.show()

# ====================== 模块4：用户评分分布（修复版） ======================
print("\n" + "="*50)
print("四、用户评分分布")
print("="*50)

# 只取有评价的订单，避免浮点精度问题导致的显示混乱
df_scored = df_analysis[df_analysis["review_score_avg"].notna()].copy()

# 按0.5分箱统计
bins = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
labels = ["1分", "2分", "3分", "4分", "5分"]
df_scored["score_bin"] = pd.cut(df_scored["review_score_avg"], bins=bins, labels=labels)

score_dist = df_scored["score_bin"].value_counts().sort_index()
score_dist_pct = (score_dist / len(df_scored) * 100).round(2)

print("评分 | 订单数 | 占比")
print("-" * 30)
for score in score_dist.index:
    print(f"{score} | {score_dist[score]:,} | {score_dist_pct[score]}%")

print(f"\n注：评分统计基于 {len(df_scored):,} 条有评价订单，"
      f"未评价订单 {(len(df_analysis)-len(df_scored)):,} 条未纳入")

plt.figure(figsize=(10, 5))
sns.countplot(data=df_scored, x="score_bin", palette="RdYlGn", order=labels)
plt.title("用户评分分布（有评价订单）", fontsize=14)
plt.xlabel("评分", fontsize=12)
plt.ylabel("订单数量", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_ROOT / "评分分布.png", dpi=300, bbox_inches='tight')
plt.show()

# ====================== 模块5：用户地域与支付分析 ======================
print("\n" + "="*50)
print("五、用户地域与支付分析")
print("="*50)

# 5.1 各州订单量TOP10
state_orders = df_analysis.groupby("customer_state").agg(
    订单量=("order_id", "count"),
    平均客单价=("order_value", "mean"),
    延迟率=("is_delay", "mean")
).reset_index().sort_values("订单量", ascending=False).head(10)

print("\n【各州订单量TOP10】")
print(state_orders.round(2).to_string(index=False))

plt.figure(figsize=(12, 5))
sns.barplot(data=state_orders, x="customer_state", y="订单量", palette="Blues_r")
plt.title("各州订单量TOP10", fontsize=14)
plt.xlabel("州缩写", fontsize=12)
plt.ylabel("订单数量", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_ROOT / "各州订单量TOP10.png", dpi=300, bbox_inches='tight')
plt.show()

# 5.2 支付方式分布
payment_dist = df_analysis["main_payment_type"].value_counts().reset_index()
payment_dist.columns = ["支付方式", "订单数"]
payment_dist["占比"] = (payment_dist["订单数"] / payment_dist["订单数"].sum() * 100).round(2)

print("\n【支付方式分布】")
print(payment_dist.to_string(index=False))

plt.figure(figsize=(8, 6))
sns.barplot(data=payment_dist, x="支付方式", y="订单数", palette="Set2")
plt.title("用户支付方式分布", fontsize=14)
plt.xlabel("支付方式", fontsize=12)
plt.ylabel("订单数量", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_ROOT / "支付方式分布.png", dpi=300, bbox_inches='tight')
plt.show()

# 5.3 分期情况统计（仅统计信用卡支付订单）
credit_orders = df_analysis[df_analysis["main_payment_type"] == "credit_card"].copy()
installment_dist = credit_orders["total_installments"].value_counts().sort_index()

print("\n【信用卡分期期数分布】")
print(f"信用卡支付订单共 {len(credit_orders):,} 单")
print(f"平均分期期数：{credit_orders['total_installments'].mean():.1f} 期")
print(f"最常见分期：{installment_dist.idxmax()} 期")

# 分期与客单价关系
installment_price = credit_orders.groupby("total_installments").agg(
    订单数=("order_id", "count"),
    平均客单价=("order_value", "mean")
).reset_index()
print("\n【分期期数 vs 客单价】")
print(installment_price.head(10).round(2).to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(x=installment_dist.index.astype(int), y=installment_dist.values, color="#3498db")
plt.title("信用卡分期期数分布", fontsize=14)
plt.xlabel("分期期数", fontsize=12)
plt.ylabel("订单数量", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_ROOT / "分期期数分布.png", dpi=300, bbox_inches='tight')
plt.show()

# ====================== 模块6：下单时间规律 ======================
print("\n" + "="*50)
print("六、用户下单时间规律")
print("="*50)

# 按小时分布
hour_dist = df_analysis["order_hour"].value_counts().sort_index()
plt.figure(figsize=(10, 4))
sns.lineplot(x=hour_dist.index, y=hour_dist.values, marker='o', color='#9b59b6', linewidth=2)
plt.title("24小时下单量分布", fontsize=14)
plt.xlabel("小时", fontsize=12)
plt.ylabel("订单数量", fontsize=12)
plt.xticks(range(0, 24))
plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(DATA_ROOT / "24小时下单分布.png", dpi=300, bbox_inches='tight')
plt.show()

# 按星期分布（0=周一，6=周日）
weekday_dist = df_analysis["order_weekday"].value_counts().sort_index()
weekday_labels = ["周一", "周二", "周三", "周四", "周五", "周六", "周日"]
plt.figure(figsize=(8, 4))
sns.barplot(x=weekday_labels, y=weekday_dist.values, palette="Pastel1")
plt.title("星期下单量分布", fontsize=14)
plt.ylabel("订单数量", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_ROOT / "星期下单分布.png", dpi=300, bbox_inches='tight')
plt.show()

# ====================== 模块7：核心指标相关性 ======================
print("\n" + "="*50)
print("七、核心数值指标相关性")
print("="*50)

corr_cols = ["order_value", "delivery_hours", "review_score_avg", "item_count", "freight_ratio"]
corr_matrix = df_analysis[corr_cols].corr()
print(corr_matrix.round(3))

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="RdBu_r", vmin=-1, vmax=1, fmt=".3f")
plt.title("核心指标相关性热力图", fontsize=14)
plt.tight_layout()
plt.savefig(DATA_ROOT / "指标相关性热力图.png", dpi=300, bbox_inches='tight')
plt.show()

# ====================== 模块8：配送时长 vs 用户评分（因果分析） ======================
print("\n" + "="*50)
print("八、配送时长 vs 用户评分")
print("="*50)

df_scored_full = df_analysis[df_analysis["review_score_avg"].notna()].copy()
df_scored_full["delivery_bin"] = pd.cut(
    df_scored_full["delivery_hours"] / 24,
    bins=[0, 7, 14, 21, 30, 999],
    labels=["≤7天", "8-14天", "15-21天", "22-30天", ">30天"]
)

delivery_impact = df_scored_full.groupby("delivery_bin", observed=True).agg(
    订单数=("order_id", "count"),
    平均评分=("review_score_avg", "mean"),
    差评率=("is_bad_review", "mean")
).reset_index()
print(delivery_impact.round(3))

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()
sns.barplot(data=delivery_impact, x="delivery_bin", y="平均评分",
            palette="RdYlGn_r", ax=ax1, alpha=0.8)
ax1.set_ylim(0, 5)
ax1.set_ylabel("平均评分", fontsize=12)
ax2.plot(range(len(delivery_impact)), delivery_impact["差评率"]*100,
         color='red', marker='o', linewidth=2, label="差评率(%)")
ax2.set_ylabel("差评率 (%)", fontsize=12, color='red')
ax2.tick_params(axis='y', labelcolor='red')
plt.title("配送时长 vs 平均评分 & 差评率", fontsize=14)
plt.xlabel("配送时长区间", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_ROOT / "配送评分关系.png", dpi=300, bbox_inches='tight')
plt.show()

# ====================== 模块9：物流延迟归因分析 ======================
print("\n" + "="*50)
print("十、物流延迟归因分析")
print("="*50)

print("\n【延迟率最高州 TOP5（订单量≥100）】")
delay_state = df_analysis.groupby("customer_state").agg(
    订单量=("order_id", "count"),
    延迟率=("is_delay", "mean"),
    平均配送天数=("delivery_hours", lambda x: x.mean()/24)
).reset_index()
print(delay_state[delay_state["订单量"] >= 100].sort_values("延迟率", ascending=False).head().round(3).to_string(index=False))

print("\n【延迟率最高品类 TOP5（订单量≥500）】")
delay_cat = df_analysis.groupby("product_category_name_english").agg(
    订单量=("order_id", "count"),
    延迟率=("is_delay", "mean"),
    平均配送天数=("delivery_hours", lambda x: x.mean()/24)
).reset_index()
print(delay_cat[delay_cat["订单量"] >= 500].sort_values("延迟率", ascending=False).head().round(3).to_string(index=False))

print("\n🎉 全部分析完成，所有图表已保存到当前目录")